# Introduction au Dataframe avec Spark


## 1 Présentation

Pour faciliter le développement de programme, Spark à partir de la version 2 intégre la notion de Dataframe.

Le DataFrame est une couche d'abstraction des RDD, qui présentent les données comme des tables de base de données sans se préoccuper de la taille des données.
Les DataFrames offrent de nombreux avantages:
* Une syntaxe beaucoup plus simple.
* Possibilité d'utiliser SQL directement dans la trame de données.
* La parallélisation des traitements dans une architecture distribuée est gérée par Spark.

Si vous avez utilisé R ou même la bibliothèque pandas avec Python, vous connaissez probablement déjà le concept des DataFrames. 

Vous êtes prêt, jouons un peu avec les dataframes ;-)

## 2 Pré-requis

### Démarrer Spark


Pour démarrer votre cluster Spark, si cela n'a pas déjà été fait, vous devrez ouvrir un terminal puis exécuter la commande suivante :

Vérifiez que les containers smaster, sworker1 et sworker2 sont démarrés :

Vérifier que vous disposez bien du répertoire dans HDFS `/data/tpspark/` :

In [ ]:
!$HADOOP_HOME/bin/hdfs dfs -ls /data/tpspark

Si ce n'est pas le cas merci de procéder aux commandes suivantes :

In [ ]:
!$HADOOP_HOME/bin/hdfs dfs -mkdir -p /data/tpspark
!$HADOOP_HOME/bin/hdfs dfs -put /home/jovyan/data/sales_info.csv /data/tpspark/
!$HADOOP_HOME/bin/hdfs dfs -put /home/jovyan/data/wash_dc_crime_incidents_2013.csv /data/tpspark/
!$HADOOP_HOME/bin/hdfs dfs -put /home/jovyan/data/appl_stock.csv /data/tpspark/

### Connexion à Spark

A partir de ce notebook, vous établirez une connexion avec le cluster Spark en python.
Votre Jupyter Notebook dispose de la librairie pyspark pour pouvoir communiquer avec un cluster SPARK.

Pour le vérifier, on peut consulter la liste des librairie python installée dans notre environnement.

In [ ]:
!pip list | grep pyspark

Pour lister les classes fournies par pyspark, utiliser simplement la commande dir en python :

In [ ]:
import pyspark
dir(pyspark)

Pour travailler avec les RDD, nous utilisions la classe SparkContext pour interagir avec notre cluster.
C'est à partir du SparkContext qu'on peut créer et manipuler des RDDs (Resilient Distributed Datasets)

Les Dataframes offrent une API de plus haut niveau que les RDD et pour travailler avec les Dataframes, Spark a simplifié ,à partir de la version 2.0, la connexion au cluster avec une nouvelles classe `SparkSession`.
C'est une API plus riche qui vous permettra de travailler avec les RDD, les Dataframe et SPARK SQL.

Pour établir une connexion à un cluster Spark, nous utiliserons à présent la classe `SparkSession`.

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.html#pyspark.sql.SparkSession

Exécutez la cellule ci-dessous pour vous connecter à notre Cluster Spark `smaster`  :

In [ ]:
# N'oubliez pas de fermer la connexion à la fin du TP
# spark.stop()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("spark://smaster:7077").appName("TPDF02").getOrCreate()


Pour vérifier que notre connexion est effective, on peut consulter notre objet `spark`:

In [ ]:
spark

Notre objet de connexion 'spark'  dispose d'un ensemble de fonctions pour analyser différentes sources de données : JSON, AVRO, PARQUET, CSV, base de données ...

* `csv(path)`
* `jdbc(url, table, ..., connectionProperties)`
* `json(path)`
* `format(source)`
* `load(path)`
* `orc(path)`
* `parquet(path)`
* `table(tableName)`
* `text(path)`
* `textFile(path)`

On pourra aussi y retrouver des fonctions pour associer des schémas de données à une source de données ou configurer les options de lectures des données :

* `option(key, value)`
* `options(map)`
* `schema(schema)`



## 1 - Les fichiers de données

Pour lire les fichiers aux formats communs tels que csv, json, parquet on peut utiliser [la méthode read de la classe `SparkSession`.](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.read.html#pyspark.sql.SparkSession.read)

#### Quelle classe d'objet est retournée par la méthode read ?

#### Quelle méthode dispose la classe en question ?

En pratique, pour lire un fichier il nous suffit donc d'utiliser la méthode read et de spécifier le format de notre fichier :

In [ ]:
df_csv = spark.read.csv("/data/tpspark/sales_info.csv", header=True)

Il est donc simple de lire des fichiers avec des formats connus.

Pour afficher le contenu, il suffira d'utiliser la méthode `show` pour afficher les 20 premières lignes de notre dataframe.

In [ ]:
df_csv.show()

#### A partir de la documentation Spark, afficher les 3 premiers éléments de votre dataframe en vertical ? 

In [ ]:
# Compléter votre réponse



## 2 - Les schémas

Examinons à présent les types associés aux colonnes de notre dataframe :

In [ ]:
df_csv.printSchema()

On peut voir que le type des colonnes est par défaut `String`. 
Il est possible de déterminer les types de nos colonnes par inférence lors du chargement des données avec la méthode inferSchema :

[https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html#pyspark.sql.DataFrameReader.csv](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html#pyspark.sql.DataFrameReader.csv)


In [ ]:
df_csv = spark.read.csv("/data/tpspark/sales_info.csv",inferSchema=True, header=True)




Regardons à présent le résultat :

In [ ]:
df_csv.printSchema()

Pas mal, Sales est à présent un Double / Float même si on préférerait utiliser un int.

Vous l'aurez compris Spark utilise un échantillonage des données pour évaluer le schema des données.
Il faut l'avouer, c'est bien plus pratique que les RDD.

Bien entendu, si le schéma ne vous convient pas vous pouvez lui indiquer les types et les noms des colonnes que vous souhaitez.

[https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.types.StructType.html#pyspark.sql.types.StructType](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.types.StructType.html#pyspark.sql.types.StructType)


In [ ]:
from pyspark.sql.types import *
# Required for StructField, StringType, IntegerType, etc.


fields = [  StructField("Compagnie", StringType(), True),
            StructField("Personne", StringType(), True),
            StructField("Ventes", IntegerType(), True)
            ]
            
csvSchema = StructType(fields)

In [ ]:
df_csv_with_my_schema = spark.read.csv("/data/tpspark/sales_info.csv",schema=csvSchema, header=True)

In [ ]:
df_csv_with_my_schema.printSchema()

In [ ]:
df_csv_with_my_schema.show()

#### Pouvez vous expliquer les valeurs `null` pour la colonne `Vente` ?

## 3 - Les formats de fichiers

Les fichiers csv et texte ne sont pas des formats adaptés pour de grosse volumétrie. 
Une bonne pratique dans les environnements de BIG DATA est de travailler avec des formats binaires pour 2 raisons :
* il consomme moins de stockage.
* il est plus facile de déplacer des données via le réseau.

Dans les écosystèmes HADOOP, les données sont sérialisées sous 2 formes :
* Ligne : Avro, Kryo, Protobuff
* Column : Parquet, Orc

En entreprise, vous trouverez souvent les formats Avro ou parquet pour leur performance.

Pour convertir un fichier en parquet c'est aussi simple que ça :

In [ ]:
df_csv_with_my_schema.write.format("parquet").save("/data/tpspark/sales_info.parquet")

Vous pouvez remarquer qu'au format parquet, notre fichier sales_info.parquet est un répertoire contenant des fichiers part-00000, cela vous rappelle-t'il quelque chose ?

En effet, pour traiter des données parquet sait que les données vont être distribuées et par conséquent applique le principe de partitionnement. Pour un fichier de plusieurs Go, il est possible d'indiquer la taille des partitions parquet.

In [ ]:
!$HADOOP_HOME/bin/hdfs dfs -ls /data/tpspark/sales_info.parquet/

Notez que la plupart des formats binaires inscrit le schéma des données dans chaque partition.
vous n'avez plus besoin de déclarer les types contrairement au fichier texte comme json ou csv.

Vérifions cela :

In [ ]:
Sales_parquet = spark.read.parquet("/data/tpspark/sales_info.parquet")

In [ ]:
Sales_parquet.printSchema()

Trop cool, non !!!

In [ ]:
Sales_parquet.show()

On peut facilement convertir un Dataframe d'un format de fichier à un autre.

#### Pouvez vous convertir le dataframe `Sales_parquet` au format json (/data/tpspark/sales.json) ? 

In [ ]:
# Compléter votre réponse


#### Vérifiez sur HDFS le contenu de votre fichier json :

In [ ]:
# Compléter votre réponse


Pour lire ensuite vos données.

In [ ]:
Sales_json = spark.read.json("/data/tpspark/sales.json")

In [ ]:
Sales_json.show()

## 4 - Représentation des données 

Les DataFrames sont une structure de données largement utilisée en Python, notamment à travers la bibliothèque Pandas.
Dans Apache Spark, on peut également convertir des DataFrames Spark en Dataframe Pandas .

Cette conversion est particulièrement utile pour visualiser graphiquement les données ou effectuer des analyses plus détaillées avec Pandas.

Avant de procéder à la conversion, assurez-vous que la bibliothèque Pandas est bien installée dans l’environnement Jupyter.
Si ce n’est pas le cas, vous pouvez l’installer avec la commande suivante : 


In [ ]:
!pip install pandas

Une fois l’installation terminée, vous pourrez facilement analyser et visualiser vos données avec Pandas

Par exemple, pour convertir le DataFrame Spark Sales_parquet en DataFrame Pandas, il suffit d’utiliser la méthode toPandas() :

In [ ]:
from pandas import DataFrame
Sales_parquet = spark.read.parquet("/data/tpspark/sales_info.parquet")
Sales_df_panda = Sales_parquet.toPandas()

In [ ]:
Sales_df_panda

C’est aussi simple que ça !
Au passage, on peut déjà voir que l'affichage est déjà plus sympa ;-).

On peut vérifier que notre Dataframe Panda dispose d'un schema similaire que notre Dataframe Spark :

In [ ]:
print(Sales_df_panda.dtypes)

In [ ]:
Sales_parquet.printSchema()

Oui, il y a une différence notable dans la gestion des types entre DataFrame Spark (pyspark.sql.DataFrame) et DataFrame Pandas (pandas.DataFrame).
Dans Spark, les types sont définis selon pyspark.sql.types, qui sont différents de ceux de Pandas.

Vous l'aurez compris le type objet n'est autre qu'une string.

De plus, on pourra remarquer que Pandas à converti la colonne Ventes en float et cela est due à la manière dont Pandas gère les valeurs nulles (NaN).
Pandas représente les integer avec des valeurs nulles(NaN) uniquement avec le type float.

Nous verrons dans la suite que pandas nous permettra de bénéficier des librairies graphiques pour explorer plus facilement nos données.

## 5 - Exploration des DataFrames

Pour manipuler les DataFrames, nous avons un ensemble de méthodes disponibles :

[https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html#pyspark.sql.DataFrame.select](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html#pyspark.sql.DataFrame.select)

In [ ]:
df = spark.read.parquet("/data/tpspark/sales_info.parquet")

Pour afficher les colonnes :

In [ ]:
df.columns

Travaillons à présent sur les selections que nous pouvons faire :

In [ ]:
dfColVentes=df.select('Ventes)

In [ ]:
type(dfColVentes)

Le résultat retourné par le select est un DataFrame

In [ ]:
dfColVentes.show()

Pour selectionner les N premières lignes, on trouvera la fonction head :

In [ ]:
# Retourne une d'object RowReturns list of Row objects
df.head(2)

Pour sélectionner multiples colonnes :

In [ ]:
df.select('*').show()

ou

In [ ]:
df.select(['Compagnie','Ventes']).show()

ou

In [ ]:
df.select(df.Compagnie, df.Ventes).show()

Pour le renommage d'une colonne on peut utiliser la méthode alias :

In [ ]:
df.select(df.Compagnie.alias('col1'), df.Ventes.alias('col2')).show()

Si on veut créer de nouvelles colonnes à partir d'une colonne existante  :

In [ ]:
df.withColumn('Ventes augmentées de 10%',df['Ventes'] * 1.1).show()

Pour renommer une colonne

In [ ]:
# Simple Rename
df.withColumnRenamed('Compagnie','Company').show()

On peut filtrer sur un critére :

In [ ]:
df.filter(df.Personne=="Sam").show()

Pour les opérations d'agrégation, on utilisera les fonctions du module pyspark.sql.functions lequel regroupe les fonctions max, min, mean ...

In [ ]:
import pyspark.sql.functions as f

df.groupBy('Compagnie').agg(f.max('Ventes').alias('Max Vente'), f.min('Ventes').alias('Min Vente'), f.mean('Ventes').alias('Moy_Vente')).orderBy('Compagnie').show()

Pour exploiter les données, une bonne pratique est de visualiser les résultats sous forme de graphes.
Python offre de nombreuses librairies graphique, nous utiliserons un exemple avec la librairie matplotlib.

In [ ]:
!pip install seaborn matplotlib


Reprenons notre exemple précédent et conservons le résultat dans un Dataframe `stat`:

In [ ]:
import pyspark.sql.functions as f

stat=df.groupBy('Compagnie').agg(f.max('Ventes').alias('Max Vente'), f.min('Ventes').alias('Min Vente'), f.mean('Ventes').alias('Moy_Vente')).orderBy('Compagnie')

L’analyse des données est effectuée par Spark, puis le driver Spark est chargé de représenter les résultats sous forme de graphiques.
Bien entendu, si la quantité de données est importante, il est recommandé d’utiliser une machine disposant de suffisamment de ressources pour assurer un affichage fluide.

Pour comparer les ventes par compagnie, on peut, par exemple, utiliser un graphique en barres comme celui-ci :


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df_pandas=stat.toPandas()

df_melted = df_pandas.melt(id_vars="Compagnie", var_name="Type Vente", value_name="Valeur")

# Création du graphique
plt.figure(figsize=(10, 6))
sns.barplot(x="Compagnie", y="Valeur", hue="Type Vente", data=df_melted, palette="Set2")

# Ajout de labels et titre
plt.xlabel("Compagnie")
plt.ylabel("Valeur des ventes")
plt.title("Comparaison des ventes par compagnie")
plt.legend(title="Type de Vente")
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Affichage
plt.show()



La représentation graphique peut être un atout majeur dans l'interprétation des données.

Au dela des aspects graphiques, vous avez aussi la possibilité de faire des jointures entre des sources de données avec différents formats.
Par exemple, créons un Dataframe qui permet d'obtenir la localisation d'une entreprise :

In [ ]:
from pyspark.sql import *
# Création d'un dataframe des localisations des différentes compagnies.
# Le code ci-dessous indique comment créer un dataframe manuellement

Location = Row("Compagnie", "location")
location1 = Location('GOOG', 'US')
location2 = Location('MSFT', 'EUROPE')
location3 = Location('FB', 'ASIA')

locationRows =[location1,location2,location3]
df_location = spark.createDataFrame(locationRows)
type(df_location)

On affiche le contenu de notre nouveau dataframe :

In [ ]:
df_location.show()

On affiche la structure du dataframe :

In [ ]:
df_location.printSchema()

On réalise la jointure entre le dataframe df listant toutes les ventes des companies et le dataframe df_location précisant la localisation des companies :

In [ ]:
df.join(df_location, df_location.Compagnie == df.Compagnie, "left_outer").show()

## 6 - Utilisation du SQL

Pour utiliser des requêtes SQL directement sur un DataFrame, vous devrez le déclarer comme une vue temporaire:

In [ ]:
# La méthode createOrReplaceTempView enregistre 
# le DataFrame comme une view temporaire
df.createOrReplaceTempView("sales")

In [ ]:
sql_sales = spark.sql("SELECT * FROM sales")

In [ ]:
sql_sales

In [ ]:
sql_sales.show()

Bien entendu, vous pouvez appliquer des requêtes SQL avec des conditions :

In [ ]:
spark.sql("SELECT * FROM sales WHERE Ventes > 500").show()

Spark SQL supporte un sous ensemble de la norme SQL92. 

Bravo !!!

Vous êtes prêt à faire vos premiers exercices avec les Dataframes.